In [1]:
import polars as pl
import pandas as pd
from lightgbm import LGBMClassifier

In [14]:
lazy_df = pl.scan_parquet('../dataset/final_train_data/final_data.parquet')

last_cols = [c for c in lazy_df.columns if "_last" in c or c == "customer_ID" or c == "target"]
print(last_cols)

cols_to_load = last_cols
df = lazy_df.select(cols_to_load).collect()

# print(df['S_2'])


C:\Users\HP\AppData\Local\Temp\ipykernel_16528\3908567723.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  last_cols = [c for c in lazy_df.columns if "_last" in c or c == "customer_ID" or c == "target"]


['customer_ID', 'target', 'P_2_last', 'D_39_last', 'B_1_last', 'B_2_last', 'R_1_last', 'S_3_last', 'D_41_last', 'B_3_last', 'D_42_last', 'D_43_last', 'D_44_last', 'B_4_last', 'D_45_last', 'B_5_last', 'R_2_last', 'D_46_last', 'D_47_last', 'D_48_last', 'D_49_last', 'B_6_last', 'B_7_last', 'B_8_last', 'D_50_last', 'D_51_last', 'B_9_last', 'R_3_last', 'D_52_last', 'P_3_last', 'B_10_last', 'D_53_last', 'S_5_last', 'B_11_last', 'S_6_last', 'D_54_last', 'R_4_last', 'S_7_last', 'B_12_last', 'S_8_last', 'D_55_last', 'D_56_last', 'B_13_last', 'R_5_last', 'D_58_last', 'S_9_last', 'B_14_last', 'D_59_last', 'D_60_last', 'D_61_last', 'B_15_last', 'S_11_last', 'D_62_last', 'D_63_last', 'D_64_last', 'D_65_last', 'B_16_last', 'B_17_last', 'B_18_last', 'B_19_last', 'D_66_last', 'B_20_last', 'D_68_last', 'S_12_last', 'R_6_last', 'S_13_last', 'B_21_last', 'D_69_last', 'B_22_last', 'D_70_last', 'D_71_last', 'D_72_last', 'S_15_last', 'B_23_last', 'D_73_last', 'P_4_last', 'D_74_last', 'D_75_last', 'D_76_last

In [17]:
all_importances = []
n_iterations = 3

defaults_df = df.filter(pl.col("target") == 1)
non_defaults_df = df.filter(pl.col("target") == 0)

for i in range(n_iterations):
    print(f"Iteration {i+1}/{n_iterations}...")
    
    sample = pl.concat([
        defaults_df.sample(n=100000, seed=i),
        non_defaults_df.sample(n=100000, seed=i)
    ])
    
   
    drop_cols = ["customer_ID", "target", "S_2"]
    X = sample.drop([c for c in drop_cols if c in sample.columns]).to_pandas()
    y = sample["target"].to_pandas()
    
    if i == 0:
        constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
        print(f"Dropping {len(constant_cols)} constant columns.")
    
    X = X.drop(columns=constant_cols)
    
    selector = LGBMClassifier(
        n_estimators=250, 
        importance_type='gain', 
        random_state=i,
        device="gpu"
    )
    selector.fit(X, y)
    
    imp = pd.DataFrame({'feature': X.columns, f'gain_{i}': selector.feature_importances_})
    all_importances.append(imp.set_index('feature'))


final_rank = pd.concat(all_importances, axis=1).mean(axis=1).sort_values(ascending=False)

top_features_all = final_rank.head(50)
print("\n--- Top Features (Audit) ---")
print(top_features_all.head(5))

magic_5 = final_rank.head(5).index.tolist()
print("\nFINAL MAGIC 5 LIST:")
print(magic_5)

import json
with open('basic_magic_5_features.json', 'w') as f:
    json.dump(magic_5, f)

Iteration 1/3...
Dropping 0 constant columns.
[LightGBM] [Info] Number of positive: 100000, number of negative: 100000
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 25674
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 187
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 97 dense feature groups (19.07 MB) transferred to GPU in 0.016471 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Iteration 2/3...
[LightGBM] [Info] Number of positive: 100000, number of negative: 100000
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 25674
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 187
[LightG